# Autoresearch: Cross-Method Comparison

Reads all `results/**/results.tsv` files and compares methods (baseline, chaos, etc.).

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

RESULTS_DIR = Path("results")

# Load all results.tsv files, tagging each with method and run_id from path
# Expected structure: results/<method>/<run_id>/results.tsv
frames = []
for tsv_path in sorted(RESULTS_DIR.glob("**/results.tsv")):
    parts = tsv_path.relative_to(RESULTS_DIR).parts
    if len(parts) < 3:
        continue
    method, run_id = parts[0], parts[1]
    df = pd.read_csv(tsv_path, sep="\t")
    df["method"] = method
    df["run_id"] = run_id
    df["val_bpb"] = pd.to_numeric(df["val_bpb"], errors="coerce")
    df["best_val_bpb"] = pd.to_numeric(df["best_val_bpb"], errors="coerce")
    df["iter"] = pd.to_numeric(df["iter"], errors="coerce")
    frames.append(df)

if not frames:
    raise FileNotFoundError(f"No results.tsv files found under {RESULTS_DIR}/")

all_df = pd.concat(frames, ignore_index=True)
methods = sorted(all_df["method"].unique())
print(f"Loaded {len(all_df)} experiments across {len(methods)} method(s): {methods}")
print(f"Runs per method: {all_df.groupby('method')['run_id'].nunique().to_dict()}")

## 1. Envelope Plot: Best val_bpb vs Iteration

The primary metric. Each line is one run; the y-axis shows the best val_bpb achieved so far at each iteration.

In [ ]:
METHOD_COLORS = {
    "baseline": "#2ecc71",
    "chaos": "#e74c3c",
}

def color_for(method):
    return METHOD_COLORS.get(method, f"C{hash(method) % 10}")

fig, ax = plt.subplots(figsize=(14, 7))

for method in methods:
    method_df = all_df[all_df["method"] == method]
    runs = sorted(method_df["run_id"].unique())
    for i, run_id in enumerate(runs):
        run_df = method_df[method_df["run_id"] == run_id].sort_values("iter")
        label = f"{method}" if i == 0 else None
        ax.plot(
            run_df["iter"], run_df["best_val_bpb"],
            color=color_for(method), alpha=0.7, linewidth=1.5,
            label=label,
        )

ax.set_xlabel("Iteration", fontsize=12)
ax.set_ylabel("Best val_bpb (lower is better)", fontsize=12)
ax.set_title("Envelope: Best val_bpb Over Time by Method", fontsize=14)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.savefig("envelope.png", dpi=150, bbox_inches="tight")
plt.show()

## 2. Terminal Performance: Best val_bpb per Run

Box plot of the best val_bpb each run achieved, grouped by method.

In [ ]:
terminal = (
    all_df.groupby(["method", "run_id"])["best_val_bpb"]
    .min()
    .reset_index()
    .rename(columns={"best_val_bpb": "terminal_bpb"})
)

fig, ax = plt.subplots(figsize=(8, 6))
data_by_method = [terminal[terminal["method"] == m]["terminal_bpb"].values for m in methods]
bp = ax.boxplot(data_by_method, labels=methods, patch_artist=True, widths=0.5)
for patch, method in zip(bp["boxes"], methods):
    patch.set_facecolor(color_for(method))
    patch.set_alpha(0.6)

# Overlay individual points
for i, method in enumerate(methods):
    vals = terminal[terminal["method"] == method]["terminal_bpb"].values
    ax.scatter(
        np.full_like(vals, i + 1) + np.random.uniform(-0.05, 0.05, len(vals)),
        vals, color=color_for(method), s=40, zorder=5, edgecolors="black", linewidths=0.5,
    )

ax.set_ylabel("Best val_bpb (lower is better)", fontsize=12)
ax.set_title("Terminal Performance by Method", fontsize=14)
ax.grid(True, alpha=0.2, axis="y")
plt.tight_layout()
plt.savefig("terminal.png", dpi=150, bbox_inches="tight")
plt.show()

print(terminal.groupby("method")["terminal_bpb"].describe())

## 3. Time to Threshold

How many iterations does each method need to reach a target val_bpb? Uses the best baseline terminal performance as the threshold.

In [ ]:
# Use the median terminal bpb of baseline as threshold (if baseline exists)
if "baseline" in methods:
    threshold = terminal[terminal["method"] == "baseline"]["terminal_bpb"].median()
else:
    threshold = terminal["terminal_bpb"].median()

print(f"Threshold: {threshold:.6f}")

# For each run, find the first iteration where best_val_bpb <= threshold
ttts = []
for (method, run_id), run_df in all_df.groupby(["method", "run_id"]):
    run_df = run_df.sort_values("iter")
    hit = run_df[run_df["best_val_bpb"] <= threshold]
    if len(hit) > 0:
        ttts.append({"method": method, "run_id": run_id, "iters_to_threshold": hit.iloc[0]["iter"]})
    else:
        ttts.append({"method": method, "run_id": run_id, "iters_to_threshold": np.nan})

ttt_df = pd.DataFrame(ttts)

fig, ax = plt.subplots(figsize=(8, 6))
data_by_method = [ttt_df[ttt_df["method"] == m]["iters_to_threshold"].dropna().values for m in methods]
bp = ax.boxplot(data_by_method, labels=methods, patch_artist=True, widths=0.5)
for patch, method in zip(bp["boxes"], methods):
    patch.set_facecolor(color_for(method))
    patch.set_alpha(0.6)

for i, method in enumerate(methods):
    vals = ttt_df[ttt_df["method"] == method]["iters_to_threshold"].dropna().values
    ax.scatter(
        np.full_like(vals, i + 1) + np.random.uniform(-0.05, 0.05, len(vals)),
        vals, color=color_for(method), s=40, zorder=5, edgecolors="black", linewidths=0.5,
    )

ax.set_ylabel("Iterations to reach threshold", fontsize=12)
ax.set_title(f"Time to Threshold (val_bpb \u2264 {threshold:.6f})", fontsize=14)
ax.grid(True, alpha=0.2, axis="y")
plt.tight_layout()
plt.savefig("time_to_threshold.png", dpi=150, bbox_inches="tight")
plt.show()

print(ttt_df.groupby("method")["iters_to_threshold"].describe())

## 4. Experiment Outcome Rates

Keep / discard / crash rates by method.

In [ ]:
all_df["status_clean"] = all_df["status"].str.strip().str.lower()

rates = (
    all_df.groupby(["method", "status_clean"])
    .size()
    .unstack(fill_value=0)
)
rates_pct = rates.div(rates.sum(axis=1), axis=0) * 100

print("Counts:")
print(rates)
print("\nPercentages:")
print(rates_pct.round(1))

fig, ax = plt.subplots(figsize=(8, 5))
rates_pct[["keep", "discard", "crash"]].plot.bar(
    ax=ax, color=["#2ecc71", "#95a5a6", "#e74c3c"], edgecolor="black", linewidth=0.5,
)
ax.set_ylabel("% of experiments", fontsize=12)
ax.set_title("Experiment Outcome Rates by Method", fontsize=14)
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.2, axis="y")
plt.tight_layout()
plt.savefig("outcome_rates.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Longest Stall (Iterations Without Improvement)

Measures how many consecutive iterations pass without a new best. Lower is better — indicates the method keeps finding improvements.

In [ ]:
stalls = []
for (method, run_id), run_df in all_df.groupby(["method", "run_id"]):
    run_df = run_df.sort_values("iter")
    best_so_far = np.inf
    streak = 0
    longest = 0
    for _, row in run_df.iterrows():
        if row["best_val_bpb"] < best_so_far:
            best_so_far = row["best_val_bpb"]
            longest = max(longest, streak)
            streak = 0
        else:
            streak += 1
    longest = max(longest, streak)
    stalls.append({"method": method, "run_id": run_id, "longest_stall": longest})

stall_df = pd.DataFrame(stalls)
print(stall_df.groupby("method")["longest_stall"].describe())

fig, ax = plt.subplots(figsize=(8, 5))
data_by_method = [stall_df[stall_df["method"] == m]["longest_stall"].values for m in methods]
bp = ax.boxplot(data_by_method, labels=methods, patch_artist=True, widths=0.5)
for patch, method in zip(bp["boxes"], methods):
    patch.set_facecolor(color_for(method))
    patch.set_alpha(0.6)

for i, method in enumerate(methods):
    vals = stall_df[stall_df["method"] == method]["longest_stall"].values
    ax.scatter(
        np.full_like(vals, i + 1, dtype=float) + np.random.uniform(-0.05, 0.05, len(vals)),
        vals, color=color_for(method), s=40, zorder=5, edgecolors="black", linewidths=0.5,
    )

ax.set_ylabel("Longest streak without improvement", fontsize=12)
ax.set_title("Longest Stall by Method", fontsize=14)
ax.grid(True, alpha=0.2, axis="y")
plt.tight_layout()
plt.savefig("stall.png", dpi=150, bbox_inches="tight")
plt.show()